In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# i load the olist tables for initial data audit
tables = {
    "customers": pd.read_csv("olist_customers_dataset.csv"),
    "geolocation": pd.read_csv("olist_geolocation_dataset.csv"),
    "order_items": pd.read_csv("olist_order_items_dataset.csv"),
    "order_payments": pd.read_csv("olist_order_payments_dataset.csv"),
    "order_reviews": pd.read_csv("olist_order_reviews_dataset.csv"),
    "orders": pd.read_csv("olist_orders_dataset.csv"),
    "products": pd.read_csv("olist_products_dataset.csv"),
    "sellers": pd.read_csv("olist_sellers_dataset.csv"),
    "category_translation": pd.read_csv("product_category_name_translation.csv")
}

# i fix the table sizes after loading to understand the data scale
print("\nTable sizes")
for name, df in tables.items():
    print(f"{name:25} rows={df.shape[0]:8} | columns={df.shape[1]:2}")

# i check data types and missing values to find potential problems in the datasets
print("\nDtypes + nulls")
for name, df in tables.items():
    print(f"\n--- {name} ---\nDtypes:\n{df.dtypes}\n\nNulls:\n{df.isna().sum()}")

# i check for full row duplicates to clean the data from repeated records
print("\nFull duplicates")
for name, df in tables.items():
    print(f"{name:25} full duplicates = {df.duplicated().sum()}")

# i check primary keys to ensure identifier uniqueness
print("\nPrimary key check")
pk_checks = {
    "customers": ["customer_id"], "orders": ["order_id"], "products": ["product_id"],
    "sellers": ["seller_id"], "order_reviews": ["review_id"],
    "category_translation": ["product_category_name"],
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"]
}
pk_results = []
for name, cols in pk_checks.items():
    df = tables[name]
    nulls = df[cols].isna().any(axis=1).sum()
    status = "Ok" if len(df) == df[cols].drop_duplicates().shape[0] and nulls == 0 else "Check"
    pk_results.append({"table": name, "pk": ", ".join(cols), "rows": len(df),
                       "unique_pk": df[cols].drop_duplicates().shape[0], "null_pk_rows": nulls, "status": status})
print(pd.DataFrame(pk_results).to_string(index=False))

# i audit customer_id and customer_unique_id to understand the specifics of these identifiers
print("\nCustomer unique id audit")
c = tables["customers"]
print(f"Total rows: {len(c)}\nUnique customer_id: {c['customer_id'].nunique()}\nUnique customer_unique_id: {c['customer_unique_id'].nunique()}")
print(f"Customer_unique_id repeated: {(c['customer_unique_id'].value_counts() > 1).sum()}")

# i check referential integrity fk -> pk to ensure correct links between tables
print("\nForeign key check")
fk_checks = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),
    ("order_items", "seller_id", "sellers", "seller_id"),
    ("order_payments", "order_id", "orders", "order_id"),
    ("order_reviews", "order_id", "orders", "order_id")
]
fk_results = []
for child, child_col, parent, parent_col in fk_checks:
    missing = set(tables[child][child_col].dropna().unique()) - set(tables[parent][parent_col].dropna().unique())
    fk_results.append({"relation": f"{child}.{child_col} -> {parent}.{parent_col}", "missing_keys": len(missing)})
print(pd.DataFrame(fk_results).to_string(index=False))

# i check the cardinality of main relationships to understand the data structure
print("\nCardinality checks")
for df_name, group_col, desc in [
    ("orders", "customer_id", "orders per customer"),
    ("order_items", "order_id", "items per order"),
    ("order_payments", "order_id", "payments per order"),
    ("order_reviews", "order_id", "reviews per order"),
    ("order_items", "product_id", "occurrences per product"),
    ("order_items", "seller_id", "items per seller")
]:
    counts = tables[df_name].groupby(group_col).size()
    print(f"{desc:30} | max: {counts.max():4} | count(>1): {(counts > 1).sum()}")

# i audit geolocation to understand the specifics of geographic data
print("\nGeolocation audit")
geo = tables["geolocation"]
zip_counts = geo.groupby("geolocation_zip_code_prefix").size()
print(f"Total rows: {len(geo)} | Unique zip prefixes: {geo['geolocation_zip_code_prefix'].nunique()}")
print(f"Max rows per zip: {zip_counts.max()} | Zips with >1 row: {(zip_counts > 1).sum()}")
print(f"Unique full rows: {geo.drop_duplicates().shape[0]} | Repeated full rows: {len(geo) - geo.drop_duplicates().shape[0]}")

# i check category translations to evaluate the dictionary quality
print("\nCategory translation audit")
cat = tables["category_translation"]
print(f"Translation rows: {len(cat)} | Unique pt categories: {cat['product_category_name'].nunique()}")
print(f"Null pt categories: {cat['product_category_name'].isna().sum()} | Duplicates: {cat['product_category_name'].duplicated().sum()}")

# i evaluate category translation coverage to see how many products lack english names
print("\nCategory translation coverage")
merged = tables["products"].merge(tables["category_translation"], on="product_category_name", how="left")
print(f"Products without english translation: {merged['product_category_name_english'].isna().sum()}")

# i form the final audit summary to record all checks
print("\nFinal data audit summary")
print("1 Tables loaded and sizes fixed\n2 Data types and nulls checked\n3 Full duplicates checked\n4 Pk candidates checked\n5 Composite pks for order_items and order_payments checked\n6 Fk -> pk checked for referential integrity\n7 Cardinality of main relationships checked\n8 Geolocation checked separately due to non-unique zip prefix\n9 Category translation checked for uniqueness and coverage\n10 Customer_unique_id specificity checked separately")

# i check review_id uniqueness linked with order_id to find possible duplicate reviews
print("\nReview id uniqueness check")
rev = tables["order_reviews"]
print(f"Total rows: {len(rev)} | Unique (review_id, order_id): {rev[['review_id', 'order_id']].drop_duplicates().shape[0]}")
dup_reviews = rev[rev["review_id"].duplicated(keep=False)].sort_values("review_id")
print(dup_reviews[["review_id", "order_id", "review_score"]].head(10))

# i check the folder where python looks for files via relative path

# i load monthly metrics calculated in postgresql
# i use ready aggregation to avoid duplicating sql logic in pandas
df = pd.read_csv("monthly_olist_sales.csv", parse_dates=["month"])

# i sort months by time before plotting for correct display
df = df.sort_values("month")

# i build the first plot: value of items in delivered orders
plt.figure(figsize=(12, 5))
plt.plot(df["month"], df["item_value_delivered"], marker="o")
plt.title("Monthly value of delivered order items")
plt.xlabel("Month")
plt.ylabel("Item value")
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# i see that the indicator grew noticeably since early 2017
# high values were observed in november 2017 and spring 2018
# in june-august 2018 the item value was slightly lower than spring peaks

# i build the second plot: number of delivered orders
# this helps me understand if the value spike is linked to order count growth or average order value change
plt.figure(figsize=(12, 5))
plt.plot(df["month"], df["delivered_orders"], marker="o")
plt.title("Monthly number of delivered orders")
plt.xlabel("Month")
plt.ylabel("Delivered orders")
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# i notice the general picture is similar: as order count grew, total item value increased
# in november 2017 the number of delivered orders spiked, and in subsequent months it fluctuated

# i load monthly delivery metrics calculated in postgresql
df_delivery = pd.read_csv("monthly_delivery_delays.csv", parse_dates=["month"])

# i order months by time for correct plotting
df_delivery = df_delivery.sort_values("month")

# i build a plot of late delivery share by month
plt.figure(figsize=(12, 5))
plt.plot(df_delivery["month"], df_delivery["late_rate_pct"], marker="o")
plt.title("Monthly late delivery rate — olist")
plt.xlabel("Order purchase month")
plt.ylabel("Late deliveries (%)")
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# i see that the share of delays fluctuates significantly from month to month
# peaks stand out in november 2017 and february-march 2018, after which the indicator drops
# no steady continuous growth is observed over the entire period

# i conclude that monthly item value in delivered olist orders grew during the studied period, though fluctuations occurred
# item value dynamics were similar to order count dynamics
# november 2017 analysis showed me that the sharp value growth is primarily linked to increased order count, not average order value growth
# the sharp drop at the end of the period is linked to data incompleteness and should not be interpreted as a sales drop
# data is currently insufficient for me to confidently confirm seasonality

# i use sql query results on the link between rating and repeat purchases
# i take values directly from the obtained result
df_repurchase = pd.DataFrame({
    "rating_group": ["Low (< 3)", "Medium (3 to < 4)", "High (4 to 5)"],
    "buyers": [14024, 7875, 73461],
    "repurchase_rate_pct": [1.61, 1.79, 2.03]
})

# i compare the repeat purchase share between rating groups
plt.figure(figsize=(9, 5))
bars = plt.bar(df_repurchase["rating_group"], df_repurchase["repurchase_rate_pct"])
plt.title("Repeat purchase rate by first-order rating")
plt.xlabel("First-order rating group")
plt.ylabel("Repeat purchase rate (%)")
plt.ylim(0, 2.5)

# i label the bars with percentage values for clarity
plt.bar_label(bars, fmt="%.2f%%", padding=3)
plt.tight_layout()
plt.show()

# i see that in the studied sample, buyers with a higher first-order rating slightly more often made repeat purchases
# a positive link between rating and repeat purchase is observed, but my analysis does not prove causality